# Landing - Bronze | CineData Analytics

**Objetivo:** ler os 5 CSVs brutos (Volume do Databricks) e a cotação do dólar (API do Banco Central) e gravá-los
como tabelas **Delta** na camada **Bronze**, **sem nenhuma alteração de conteúdo ou estrutura**.

**Regras desta camada (Arquitetura Medalhão):**
1. Todas as colunas dos CSVs são lidas como `STRING` (sem `inferSchema`). A tipagem é responsabilidade da Silver.
2. Cada tabela recebe apenas a coluna técnica `ingestion_datetime` (momento da inserção).
3. Gravação em **Delta** no modo **append** — o histórico de cargas é preservado. Reexecuções geram linhas
   repetidas (com `ingestion_datetime` diferente); a **deduplicação é feita na Silver** usando essa coluna.
4. Ingestão de API: cotação do dólar PTAX (Banco Central) → `bronze.tb_cotacao_dolar`.

In [0]:
import re
import time
import requests
from datetime import date, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

dbutils.widgets.text("catalog", "workspace", "1. Catálogo (vazio = padrão da sessão)")
dbutils.widgets.text("volume_path", "/Volumes/workspace/landing/inputs", "2. Caminho do Volume com os CSVs")
dbutils.widgets.text("data_inicio", "", "3. Data início da cotação (MM-DD-AAAA) | vazio = hoje-7")
dbutils.widgets.text("data_fim", "", "4. Data fim da cotação (MM-DD-AAAA) | vazio = hoje")
dbutils.widgets.text("cotacao_manual", "", "5. Cotação manual (usada SOMENTE se a API estiver indisponível)")

CATALOG = dbutils.widgets.get("catalog").strip()
VOLUME_PATH = dbutils.widgets.get("volume_path").strip().rstrip("/")

if CATALOG:
    spark.sql(f"USE CATALOG `{CATALOG}`")

spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
print(f"Catálogo: {CATALOG or '(padrão)'} | Volume: {VOLUME_PATH}")

Catálogo: workspace | Volume: /Volumes/workspace/landing/inputs


## 1. Ingestão dos 5 arquivos CSV

* **Sem alterações de conteúdo:** tudo é lido como texto; os valores sujos (formatos de data diferentes, símbolos de moeda,
  colunas deslocadas etc.) chegam à Bronze exatamente como estão na origem.
* `multiLine=True` + `escape='"'`: sinopses e comentários podem ter quebras de linha e aspas dentro do campo — sem isso
  o Spark quebraria um registro em vários.
* O separador é detectado automaticamente pela linha de cabeçalho (`,` ou `;`), o que evita falhas caso algum arquivo
  tenha sido exportado com outro delimitador.

In [0]:
ARQUIVOS_PARA_TABELAS = {
    "movies_info_TMDB_IMDB.csv":       "bronze.tb_movies_info",
    "movies_financials_IMDB_TMDB.csv": "bronze.tb_movies_financials",
    "movies_metrics_IMDB_TMDB.csv":    "bronze.tb_movies_metrics",
    "credits_and_tags_IMDB_TMDB.csv":  "bronze.tb_credits_and_tags",
    "movies_reviews.csv":              "bronze.tb_movies_reviews",
}

def detectar_separador(caminho: str) -> str:
    """Escolhe o delimitador mais frequente na linha de cabeçalho."""
    cabecalho = spark.read.text(caminho).limit(1).collect()[0][0]
    candidatos = [",", ";", "\t", "|"]
    return max(candidatos, key=lambda s: cabecalho.count(s))

def ler_csv_bruto(caminho: str):
    """Lê o CSV com TODAS as colunas como STRING (sem inferSchema) para não alterar o conteúdo."""
    sep = detectar_separador(caminho)
    return (
        spark.read.format("csv")
        .option("header", True)
        .option("sep", sep)
        .option("multiLine", True)      
        .option("quote", '"')
        .option("escape", '"')          
        .option("inferSchema", False)   
        .option("mode", "PERMISSIVE")   
        .load(caminho)
    )

def sanitizar_nomes_colunas(df):
    """
    O Delta rejeita alguns caracteres em nomes de colunas (espaço, vírgula, ; { } ( ) = \n \t).
    Só renomeia se for ESTRITAMENTE necessário (também remove BOM e espaços nas pontas do cabeçalho).
    """
    for c in df.columns:
        novo = re.sub(r"[ ,;{}()\n\t=]", "_", c.replace("\ufeff", "").strip())
        if novo != c:
            df = df.withColumnRenamed(c, novo)
    return df

def gravar_bronze(df, tabela: str):
    """Adiciona ingestion_datetime e grava em Delta no modo APPEND."""
    (
        df.withColumn("ingestion_datetime", F.current_timestamp())  
          .write.format("delta")
          .mode("append")                     
          .option("mergeSchema", "true")
          .saveAsTable(tabela)
    )

for arquivo, tabela in ARQUIVOS_PARA_TABELAS.items():
    caminho = f"{VOLUME_PATH}/{arquivo}"
    df_bruto = sanitizar_nomes_colunas(ler_csv_bruto(caminho))
    gravar_bronze(df_bruto, tabela)
    print(f"✔ {arquivo:<36} -> {tabela:<32} | colunas lidas: {len(df_bruto.columns)}")

✔ movies_info_TMDB_IMDB.csv            -> bronze.tb_movies_info            | colunas lidas: 10
✔ movies_financials_IMDB_TMDB.csv      -> bronze.tb_movies_financials      | colunas lidas: 3
✔ movies_metrics_IMDB_TMDB.csv         -> bronze.tb_movies_metrics         | colunas lidas: 6
✔ credits_and_tags_IMDB_TMDB.csv       -> bronze.tb_credits_and_tags       | colunas lidas: 9
✔ movies_reviews.csv                   -> bronze.tb_movies_reviews         | colunas lidas: 4


## 2. Ingestão via API — cotação do dólar (PTAX / Banco Central)

* Formato de data exigido pela API: **MM-DD-AAAA**.
* Se os widgets `data_inicio`/`data_fim` estiverem vazios, usa os **últimos 7 dias corridos** a partir da data de execução
  (a API não retorna cotação em finais de semana e feriados — 7 dias garantem ao menos um dia útil).
* A API pode retornar **várias cotações por dia** (boletins de abertura, intermediários e fechamento). Todas são gravadas
  na Bronze; a Silver escolhe a última do dia.
* **Resiliência:** 3 tentativas com espera crescente. Se a API estiver inacessível (ex.: bloqueio de rede no workspace),
  o notebook usa o histórico já existente na Bronze ou a `cotacao_manual` informada — e só falha se não houver nenhuma alternativa.

In [0]:
TABELA_COTACAO = "bronze.tb_cotacao_dolar"

def definir_periodo():
    """Período da consulta no formato MM-DD-AAAA (padrão: últimos 7 dias corridos)."""
    hoje = date.today()
    inicio = dbutils.widgets.get("data_inicio").strip() or (hoje - timedelta(days=7)).strftime("%m-%d-%Y")
    fim = dbutils.widgets.get("data_fim").strip() or hoje.strftime("%m-%d-%Y")
    return inicio, fim

def buscar_cotacoes(inicio: str, fim: str, tentativas: int = 3):
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
        "CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
        f"?@dataInicial='{inicio}'&@dataFinalCotacao='{fim}'"
        "&$select=dataHoraCotacao,cotacaoCompra&$format=json"
    )
    ultimo_erro = None
    for i in range(tentativas):
        try:
            resp = requests.get(url, timeout=30)
            resp.raise_for_status()
            return resp.json().get("value", [])
        except Exception as e:                       
            ultimo_erro = e
            time.sleep(2 * (i + 1))                  
    raise RuntimeError(f"API do Banco Central indisponível após {tentativas} tentativas: {ultimo_erro}")

data_ini, data_fim = definir_periodo()
print(f"Consultando cotação PTAX de {data_ini} até {data_fim} ...")

registros = []
try:
    registros = buscar_cotacoes(data_ini, data_fim)
    print(f"API retornou {len(registros)} cotações.")
except Exception as erro:
    print(f"⚠ {erro}")
    manual = dbutils.widgets.get("cotacao_manual").strip().replace(",", ".")
    ja_existe = spark.catalog.tableExists(TABELA_COTACAO) and spark.table(TABELA_COTACAO).limit(1).count() > 0
    if manual:
        registros = [{"dataHoraCotacao": f"{date.today():%Y-%m-%d} 00:00:00.000", "cotacaoCompra": float(manual)}]
        print(f"Usando cotação manual = {manual}")
    elif ja_existe:
        print("Mantendo o histórico de cotações já existente na Bronze (nenhuma linha nova).")
    else:
        raise  

if registros:
    schema_cotacao = StructType([
        StructField("dataHoraCotacao", StringType(), True),
        StructField("cotacaoCompra", DoubleType(), True),
    ])
    linhas = [(r["dataHoraCotacao"], float(r["cotacaoCompra"])) for r in registros if r.get("cotacaoCompra") is not None]
    df_cotacao = spark.createDataFrame(linhas, schema_cotacao)
    gravar_bronze(df_cotacao, TABELA_COTACAO)
    print(f"✔ {len(linhas)} linhas gravadas em {TABELA_COTACAO}")
elif not spark.catalog.tableExists(TABELA_COTACAO):
    raise RuntimeError("Nenhuma cotação disponível: a API não retornou dados para o período e não há histórico.")

Consultando cotação PTAX de 09-12-2026 até 09-19-2026 ...
API retornou 5 cotações.
✔ 5 linhas gravadas em bronze.tb_cotacao_dolar


## 3. Conferência da carga
Contagem de linhas por tabela e a última carga registrada (`ingestion_datetime`).

In [0]:
tabelas = list(ARQUIVOS_PARA_TABELAS.values()) + [TABELA_COTACAO]
resumo = [
    (t, spark.table(t).count(), spark.table(t).agg(F.max("ingestion_datetime")).first()[0])
    for t in tabelas
]
display(spark.createDataFrame(resumo, ["tabela_bronze", "qtd_linhas", "ultima_ingestao"]))
display(spark.table(TABELA_COTACAO).orderBy(F.col("dataHoraCotacao").desc()).limit(10))

tabela_bronze,qtd_linhas,ultima_ingestao
bronze.tb_movies_info,213192,2026-09-19T12:49:00.357Z
bronze.tb_movies_financials,212330,2026-09-19T12:49:04.368Z
bronze.tb_movies_metrics,206144,2026-09-19T12:49:08.208Z
bronze.tb_credits_and_tags,211216,2026-09-19T12:49:11.546Z
bronze.tb_movies_reviews,64824,2026-09-19T12:49:14.823Z
bronze.tb_cotacao_dolar,5,2026-09-19T12:49:18.800Z


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-18 13:03:34.742036,5.1569,2026-09-19T12:49:18.800Z
2026-09-17 13:03:21.858212,5.1515,2026-09-19T12:49:18.800Z
2026-09-16 13:05:30.35873,5.152,2026-09-19T12:49:18.800Z
2026-09-15 13:09:19.199664,5.1484,2026-09-19T12:49:18.800Z
2026-09-14 13:10:08.144425,5.169,2026-09-19T12:49:18.800Z
